# Подготовим и обучим модель для работы с учебными текстами

### Установим зависимости

In [63]:
!pip install pypdf2 sentence-transformers faiss-cpu openrouter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 19.9 MB/s eta 0:00:00


## Извлечем текст из pdf.

In [2]:
import os
from PyPDF2 import PdfReader

def extract_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text.strip()

filenames = [f for f in os.listdir() if f.endswith(".pdf")]
print(f"Найдено PDF: {len(filenames)}")

doc_texts = {}
for fn in filenames:
    path = os.path.join('', fn)
    doc_texts[fn] = extract_text(path)
    print(f"Извлечён {fn}: {len(doc_texts[fn])} символов")

Найдено PDF: 21
Извлечён 1.4.12 help и man.pdf: 892 символов
Извлечён 1.4.1 Работа в командной строке.pdf: 3617 символов
Извлечён Docker.pdf: 5802 символов
Извлечён 1.4.11 Справочные системы.pdf: 5775 символов
Извлечён Основы Linux_1.pdf: 39924 символов
Извлечён Основы Linux.pdf: 8101 символов
Извлечён AAA в Windows.pdf: 6950 символов
Извлечён 1.3.3 Псевдотерминалы.pdf: 2438 символов
Извлечён Инфраструктурные сервисы в домене.pdf: 6208 символов
Извлечён 1.4.9 traceroute.pdf: 5075 символов
Извлечён 1.4.3 ifconfig.pdf: 2722 символов
Извлечён 1.3.2 Виртуальные терминалы.pdf: 2640 символов
Извлечён 1.4.10 history.pdf: 2087 символов
Извлечён Основы Linux_2.pdf: 31438 символов
Извлечён материал по дистрибутивам Linux и тп.pdf: 36372 символов
Извлечён Windows server.pdf: 1280 символов
Извлечён 1.3.1 Командный интерфейс.pdf: 5986 символов
Извлечён 1.4.2 ping.pdf: 2655 символов
Извлечён 1.4.13 Управление сетью.pdf: 14804 символов
Извлечён AD.pdf: 5350 символов
Извлечён 1.4.4 ip a.pdf: 4565 симв

## Теперь разобьем текст на чанки.

In [3]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

documents = []
doc_ids = []
metadatas = []
for fn, text in doc_texts.items():
    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        doc_id = f"{fn}_{i}"
        documents.append(chunk)
        doc_ids.append(doc_id)
        metadatas.append({"file": fn, "chunk_index": i})

print(f"Всего чанков: {len(documents)}")

Всего чанков: 441


## Построим эмбеддинги при помощи all-MiniLM-L6-v2.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(documents, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')
print(f"Форма эмбеддингов: {embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Форма эмбеддингов: (441, 384)


## Реализуем хранение эмбеддингов во внутреннем векторном хранилище при помощи FAISS.

In [ ]:
import faiss
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # простой индекс по L2-расстоянию
index.add(embeddings)
print(f"FAISS индекс создан. Размерность: {dimension}, векторов: {index.ntotal}")

FAISS индекс создан. Размерность: 384, векторов: 441


## апишем функцию поиска релевантных фрагментов по запросу пользователей (top-k).

In [5]:
def retrieve(query, embedder, index, documents, doc_ids, metadatas, top_k=5):
    """
    Возврат списка кортежей (doc_id, текст чанка, метаданные, расстояние L2).
    """
    query_emb = embedder.encode([query]).astype('float32')
    distances, indices = index.search(query_emb, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append((
            doc_ids[idx],
            documents[idx],
            metadatas[idx],
            float(dist)
        ))
    return results

## Напишем функцию формирования промпта.

### Систсемный промпт будет выглядеть так:
```
"Ты — ассистент, отвечающий строго на основе предоставленного контекста. "
    "Контекст состоит из фрагментов документов. Твои шаги:\n"
    "1. Прочитай внимательно каждый фрагмент.\n"
    "2. Найди конкретные факты, связанные с вопросом.\n"
    "3. Составь краткий ответ, используя только эти факты (можно цитировать).\n"
    "4. Если в контексте недостаточно информации для ответа, "
    "напиши ровно: 'Информации в предоставленных документах недостаточно.'\n"
    "Не добавляй ничего от себя."

  ```

In [43]:
SYSTEM_PROMPT = (
    "Ты — ассистент, отвечающий строго на основе предоставленного контекста. "
    "Контекст состоит из фрагментов документов. Твои шаги:\n"
    "1. Прочитай внимательно каждый фрагмент.\n"
    "2. Найди конкретные факты, связанные с вопросом.\n"
    "3. Составь краткий ответ, используя только эти факты (можно цитировать).\n"
    "4. Если в контексте недостаточно информации для ответа, "
    "напиши ровно: 'Информации в предоставленных документах недостаточно.'\n"
    "Не добавляй ничего от себя."
)

def build_prompt(query, retrieved_chunks):
    context_parts = []
    for i, (doc_id, chunk_text, meta, _) in enumerate(retrieved_chunks):
        context_parts.append(f"[Фрагмент {i+1}] (источник: {meta['file']})\n{chunk_text}")
    context = "\n\n".join(context_parts)

    prompt = f"{SYSTEM_PROMPT}\n\nКонтекст:\n{context}\n\nВопрос: {query}\nОтвет:"
    return prompt

## Подключим модель через OpenRouter.

In [84]:
from openrouter import OpenRouter
import os

# запрос ключа, если ещё не задан
if OPENROUTER_API_KEY not in globals():
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Введите ваш OpenRouter API key: ")
    OPENROUTER_API_KEY = OPENROUTER_API_KEY.strip()
    print("Ключ сохранён.")
else:
    print("Ключ уже в памяти.")

def generate_answer(messages, model="z-ai/glm-4.5-air:free", max_tokens=2048, temperature=0.2):
    """
    Отправляет запрос к OpenRouter и возвращает ответ модели.
    messages – список [{"role": "user", "content": "..."}]
    """
    if not OPENROUTER_API_KEY:
        return "Ошибка: API-ключ не задан."

    try:
        with OpenRouter(api_key=OPENROUTER_API_KEY) as client:
            response = client.chat.send(
                model=model,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return response.choices[0].message.content
    except Exception as e:
        return f"Ошибка при вызове LLM: {str(e)}"

print("Функция generate_answer готова.")

Введите ваш OpenRouter API key: ··········
Ключ сохранён.
Функция generate_answer готова.


## Теперь посмотрим правильно ли модель действует на тестовом промпте.

In [91]:
import faiss
import numpy as np

# проверка индекса
if 'index' not in globals() and 'index' not in locals():
    if 'embeddings' in globals() or 'embeddings' in locals():
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embeddings)
        print(f"FAISS индекс пересоздан. Размерность: {dimension}, векторов: {index.ntotal}")
    else:
        print("Ошибка:embeddings не найдены. Запустите предыдущие ячейки.")

# проверка ключа
if 'OPENROUTER_API_KEY' not in globals():
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Введите ваш OpenRouter API key: ")
    print("Ключ сохранён.")

if ('index' in globals() or 'index' in locals()) and 'OPENROUTER_API_KEY' in globals():
    question = "Что такое man? Что такое help?"
    retrieved = retrieve(question, embedder, index, documents, doc_ids, metadatas, top_k=5)
    prompt_text = build_prompt(question, retrieved)
    messages = [{"role": "user", "content": prompt_text}]

    answer = generate_answer(messages)
    print("Ответ:", answer)
    print("\nИсточники:")
    for doc_id, text, meta, dist in retrieved:
        print(f"  {meta['file']} (L2={dist:.4f})")
else:
    print("Невозможно продолжить: нет FAISS индекса или API-ключа.")

Ответ: На основе предоставленных документов:

1.  **man**:
    *   Это команда для вывода справочной информации (Фрагмент 5: "man [раздел] [название_страницы]").
    *   Справка разделена на несколько разделов (Фрагмент 5: "Вся справка разделена на несколько разделов").
    *   Название страницы обычно совпадает с именем команды или программы (Фрагмент 5: "Как правило название страницы совпадает с именем команды или программы").
    *   Используется для получения информации о командах, например `man useradd` (Фрагмент 2: "Теперь запустим help и man для useradd").

2.  **help**:
    *   Это команда, используемая для получения информации (Фрагмент 2: "Запустим help и man, чтобы узнать как проверить доступность сервера").
    *   Используется для проверки доступности сервера (Фрагмент 2: "Для проверки доступности сервера можно отправить несколько пакетов").
    *   Используется для получения информации о командах, например `help useradd` (Фрагмент 2: "Теперь запустим help и man для userad